In [7]:
import pandas as pd
import numpy as np

# ======================
# 1. LOAD DATA
# ======================
# 'patients' ya estaba en minúsculas, lo mantenemos igual
patients = pd.read_csv("dataset_p1/2mimic/PATIENTS.csv", usecols=[
    "subject_id", "gender", "dob", "dod"
])

# Cambiamos las columnas de 'admissions' a minúsculas
admissions = pd.read_csv("dataset_p1/2mimic/ADMISSIONS.csv", usecols=[
    "subject_id", "hadm_id", "admittime", "dischtime", 
    "admission_type", "hospital_expire_flag"
])

# Cambiamos las columnas de 'icustays' a minúsculas
icustays = pd.read_csv("dataset_p1/2mimic/ICUSTAYS.csv", usecols=[
    "subject_id", "hadm_id", "icustay_id", 
    "intime", "outtime"
])

# ======================
# 2. MERGE
# ======================
df = admissions.merge(patients, on="subject_id", how="inner")
df = df.merge(icustays, on=["subject_id", "hadm_id"], how="inner")

print("After merge:", df.shape)

# ======================
# 3. DATETIME CONVERSION
# ======================
date_cols = ["admittime", "dischtime", "dob", "intime", "outtime"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# ======================
# 4. FEATURE ENGINEERING
# ======================

# Edad
df["age"] = (df["admittime"] - df["dob"]).dt.days / 365

# Filtrar edades irreales (anonimización)
df = df[(df["age"] >= 0) & (df["age"] < 100)]

# Estancia hospitalaria
df["los_hosp"] = (df["dischtime"] - df["admittime"]).dt.days

# Estancia UCI
df["los_icu"] = (df["outtime"] - df["intime"]).dt.days

# ======================
# 5. LIMPIEZA
# ======================
df = df.dropna(subset=[
    "age", "los_hosp", "los_icu", "gender", "admission_type"
])

# ======================
# 6. TARGET
# ======================
df["target"] = df["hospital_expire_flag"]

# ======================
# 7. ENCODING
# ======================

# Género
df["gender"] = df["gender"].map({"M": 0, "F": 1})

# Tipo de admisión (one-hot)
df = pd.get_dummies(df, columns=["admission_type"], drop_first=True)

# ======================
# 8. FEATURE SELECTION
# ======================
feature_cols = ["age", "los_hosp", "los_icu", "gender"] + \
               [col for col in df.columns if col.startswith("admission_type_")]

X = df[feature_cols]
y = df["target"]

print("Final dataset shape:", X.shape)
print("Target distribution:\n", y.value_counts())

# ======================
# 9. QUICK SANITY CHECK
# ======================
print("\nSample data:")
print(X.head())

After merge: (136, 12)
Final dataset shape: (127, 6)
Target distribution:
 target
0    85
1    42
Name: count, dtype: int64

Sample data:
         age  los_hosp  los_icu  gender  admission_type_EMERGENCY  \
0  70.682192         8        1       1                      True   
1  36.213699        13       13       1                      True   
2  87.142466         2        2       1                      True   
3  73.726027         8        2       1                      True   
4  48.931507         0        1       0                      True   

   admission_type_URGENT  
0                  False  
1                  False  
2                  False  
3                  False  
4                  False  
